# Чат-бот на базе rugpt3small_based_on_gpt2

## Среда

### Установка бибилиотек

In [1]:
# pip install transformers datasets torch rouge detoxify nltk bert-score

### Стандартные бибилотеки

In [2]:
import time
import re
import json
import threading
import sys
import itertools

### Обработка данных

In [3]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

### Нейросеть

In [4]:
import torch
from torch.cuda.amp import GradScaler
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel, 
    GPT2Config,
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

### Визуализация

In [5]:
from tqdm.notebook import tqdm

### Метрики

In [6]:
from rouge import Rouge
from rouge_score import rouge_scorer
from detoxify import Detoxify
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score

## Замеры времени

In [7]:
# Фрейм для замеров времени
time_metrics = pd.DataFrame(columns=["stage", "time_sec", "time_formatted"])

In [8]:
# Функция для форматирования времени
def format_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds - int(seconds)) * 100)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:02}"

In [9]:
# Функция для добавления времени
def add_time_metric(stage_name, time_value):
    formatted_time = format_time(time_value)
    new_row = pd.DataFrame({
        "stage": [stage_name],
        "time_sec": [round(time_value, 2)],
        "time_formatted": [formatted_time]
    })
    global time_metrics
    time_metrics = pd.concat([time_metrics, new_row], ignore_index=True)

In [10]:
start_total_time = time.time()

## Данные

### Загрузка датасета

In [11]:
dataset = load_dataset("MLNavigator/russian-retrieval")

### Очистка датасета

In [12]:
# Очистка датасета от меток SOURCE
def clean_dataset(example):
    example['q'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['q'], flags=re.IGNORECASE).strip()
    example['a'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['a'], flags=re.IGNORECASE).strip()
    return example

dataset = dataset.map(clean_dataset)

### Уменьшение датасета для ускорения тестирования

In [13]:
dataset['train'] = dataset['train'].select(range(10000))

### Предобработка

In [14]:
# Разделение на train/val/test (80%/10%/10%)
split_dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_val_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train_data = split_dataset['train']
val_data = train_val_split['train']
test_data = train_val_split['test']

# Подготовка данных
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Установка pad_token

def preprocess_function(examples):
    inputs = [f"Вопрос: {q} Ответ: {a}" for q, a in zip(examples['q'], examples['a'])]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_attention_mask=True  # Добавить маску внимания
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Предобработка на данных
train_data = train_data.map(preprocess_function, batched=True)
val_data = val_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

## Модель

### Загрузка пердобученной модели

In [15]:
model = GPT2LMHeadModel.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

### Оптимизация для GPU или CPU

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
device

'cuda'

In [17]:
# Очистка кэша PyTorch
torch.cuda.empty_cache()

### Настройка параметров обучения с учётом параметров RTX2060

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    fp16=True if device == "cuda" else False,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to= "none",
)

### Обучение модели

In [19]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 не использует masked language modeling
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,  # data_collator вместо tokenizer
)

In [20]:
start_train_time = time.time()

In [21]:
trainer.train()
trainer.save_model("results/final_model")

Step,Training Loss,Validation Loss
200,2.395700,2.318995
400,2.277900,2.228913
600,1.957300,2.219908
800,1.875500,2.194582
1000,1.895300,2.158732
1200,1.635500,2.199235
1400,1.588000,2.182719
1600,1.383100,2.239857
1800,1.435200,2.213822
2000,1.466500,2.210358


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [22]:
end_train_time = time.time()
train_duration = end_train_time - start_train_time
print(f"Обучение заняло: {train_duration:.2f} сек")
add_time_metric("Обучение", train_duration)

Обучение заняло: 1742.69 сек


In [23]:
# Сохранения логов обучения
training_logs = trainer.state.log_history

# Сохранение в DataFrame и CSV
df_logs = pd.DataFrame([log for log in training_logs if 'loss' in log or 'eval_loss' in log])
df_logs.to_csv("rugpt_training_logs.csv", index=False)

print("Результаты обучения сохранены в rugpt_training_logs.csv")
df_logs

Результаты обучения сохранены в rugpt_training_logs.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,10.6456,234.574219,3.000000e-06,0.02,10,NaN,NaN,NaN,NaN
1,8.1544,83.102623,8.000000e-06,0.04,20,NaN,NaN,NaN,NaN
2,5.6276,84.536194,1.300000e-05,0.06,30,NaN,NaN,NaN,NaN
3,4.3176,60.482807,1.800000e-05,0.08,40,NaN,NaN,NaN,NaN
4,3.3843,26.950960,2.300000e-05,0.10,50,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
257,1.2818,14.919763,9.166667e-07,4.92,2460,NaN,NaN,NaN,NaN
258,1.2872,14.964849,7.083333e-07,4.94,2470,NaN,NaN,NaN,NaN
259,1.2934,16.918667,5.000000e-07,4.96,2480,NaN,NaN,NaN,NaN
260,1.2554,13.371373,2.916667e-07,4.98,2490,NaN,NaN,NaN,NaN


### Генерация ответа

In [24]:
# Функиця генерации ответа
def generate_answer(question):
    try:
        model.eval()
        input_text = f"Вопрос: {question} Ответ:"
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            return_attention_mask=True  # Генерация маски внимания
        ).to(device)

        start = time.time() # время начала ответа
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],  # Передача маски внимания
                max_new_tokens=50,
                num_return_sequences=1,
                no_repeat_ngram_size=2,
                num_beams=5,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        clean_response = re.sub(r'\n+.*', '', response, flags=re.IGNORECASE)
        generated_answer = clean_response.split('Ответ:')[-1].strip()

        latency = time.time() - start # длительность ответа
        
        return generated_answer, latency
            
    except Exception as e:
        print(f"Ошибка генерации: {e}")
        return "Не удалось сгенерировать ответ", 0.0 

# Функция генерации и сохранения ответов на тестовом датасете
def generate_answers_and_save(test_data, output_file="rugpt_generated_answers.json"):
    generated_data = []
    
    for example in tqdm(test_data, desc="Генерация ответов", unit="example", total=len(test_data)):
        q = example['q']
        true_answer = example['a']
        
        # Генерация ответа
        answer, latency = generate_answer(q)
        
        # Запись для сохранения
        generated_data.append({
            "Вопрос": q,
            "Эталонный ответ": true_answer,
            "Сгенерированный ответ": answer,
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Сохранение в файл
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(generated_data, f, ensure_ascii=False, indent=2)
    print(f"Сгенерированные ответы сохранены в {output_file}")

In [25]:
start_gen_time = time.time()

In [26]:
# Запуск генерации ответов
generate_answers_and_save(test_data)

Генерация ответов:   0%|          | 0/1000 [00:00<?, ?example/s]

Сгенерированные ответы сохранены в rugpt_generated_answers.json


In [27]:
end_gen_time = time.time()
gen_duration = end_gen_time - start_gen_time
print(f"Генерация ответов заняла: {gen_duration:.2f} сек")
add_time_metric("Генерация ответов", gen_duration)

Генерация ответов заняла: 952.41 сек


## Оценка качества модели

### Загрузка моделей

In [28]:
# Загрузка модели Detoxify для оценки токсичности ответов
detox = Detoxify('original', device=device)
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Функции для расчета метрик

In [29]:
# Перплексия для оценки предсказания текста
def calculate_perplexity(question, generated_answer):
    inputs = tokenizer(f"Вопрос: {question} Ответ: {generated_answer}", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

In [30]:
def calculate_bertscore(generated, true_answer):
    P, R, F1 = score(
        [generated], 
        [true_answer], 
        lang="ru", 
        model_type="bert-base-multilingual-cased",
        verbose=False
    )
    return F1.item()

In [31]:
# Токсичность ответов
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Расчет метрик из сохраненных данных

In [32]:
# Функция расчёта метрик 
def calculate_metrics_from_file(input_file="rugpt_generated_answers.json", output_file="rugpt_metrics.csv"):
    with open(input_file, "r", encoding="utf-8") as f:
        generated_data = json.load(f)
    
    metrics = []
    
    for example in tqdm(generated_data, desc="Расчет метрик", unit="example", total=len(generated_data)):
        q = example["Вопрос"]
        true_answer = example["Эталонный ответ"]
        generated = example["Сгенерированный ответ"]
        latency = example.get("Время отклика (сек)", 0.0)
        
        # Проверка, что на входе строка
        if not isinstance(generated, str):
            generated = " ".join(generated)
        
        # Расчет метрик
        ppl = calculate_perplexity(q, generated) 
        bert_score = calculate_bertscore(generated, true_answer)
        toxicity = check_toxicity(generated)
        
        metrics.append({
            "Вопрос": q,
            "Сгенерированный ответ": generated,
            "Эталонный ответ": true_answer,
            "Perplexity": round(ppl, 2),
            "BERTScore F1": round(bert_score, 4),
            "Токсичность": round(toxicity, 4),
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Создание DataFrame и сохранение
    df = pd.DataFrame(metrics)
    df.to_csv(output_file, index=False)
    print(f"Метрики сохранены в {output_file}")

    avg_latency = df["Время отклика (сек)"].mean()

    # Сводка по средним значениям
    summary = df.mean(numeric_only=True)
    print("Средние значения метрик:")
    print(summary)

In [33]:
start_metrics_time = time.time()

### Расчёт метрик

In [34]:
calculate_metrics_from_file()

Расчет метрик:   0%|          | 0/1000 [00:00<?, ?example/s]

Метрики сохранены в rugpt_metrics.csv
Средние значения метрик:
Perplexity             5.601790
BERTScore F1           0.677805
Токсичность            0.002279
Время отклика (сек)    0.948015
dtype: float64


In [35]:
end_metrics_time = time.time()
metrics_duration = end_metrics_time - start_metrics_time
print(f"Расчет метрик занял: {metrics_duration:.2f} сек")
add_time_metric("Расчёт метрик", metrics_duration)

Расчет метрик занял: 1281.29 сек


In [36]:
end_total_time = time.time()
total_duration = end_total_time - start_total_time
add_time_metric("Общее", total_duration)

In [37]:
time_metrics

,stage,time_sec,time_formatted
0,Обучение,1742.69,00:29:02.69
1,Генерация ответов,952.41,00:15:52.40
2,Расчёт метрик,1281.29,00:21:21.28
3,Общее,3990.94,01:06:30.93


## Чат-бот

In [39]:
def chat_bot():
    print("Бот: Здравствуйте! Для выхода введите 'выход'.")
    qa_history = []
    
    # Анимация ожидания
    def loading_animation(stop_event):
        symbols = itertools.cycle(['⠇', '⠋', '⠙', '⠸', '⠴', '⠦'])
        while not stop_event.is_set():
            sys.stdout.write(f"\rБот: Думаю... {next(symbols)}")
            sys.stdout.flush()
            time.sleep(0.1)
        sys.stdout.write("\r" + " " * 40 + "\r")  # Очистка строки
    
    while True:
        original_question = input("\nВы: ").strip()
        if original_question.lower() == "выход":
            break
        
        current_question = original_question
        attempts = []
        max_attempts = 3
        success = False
        clarification_used = False  # Метка, что использовалось уточнение
        
        for attempt_num in range(1, max_attempts + 1):
            # Запуск анимации
            stop_animation = threading.Event()
            animation_thread = threading.Thread(target=loading_animation, args=(stop_animation,))
            animation_thread.start()
            
            try:
                answer, latency = generate_answer(current_question)
                clean_answer = answer.split("Ответ:")[-1].strip()
                
                # Остановка анимации
                stop_animation.set()
                animation_thread.join()
                
                # Сохранение попытки
                attempts.append({
                    "attempt": attempt_num,
                    "question": current_question,
                    "answer": clean_answer,
                    "latency": round(latency, 4),
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                })
                
                print(f"Бот: {clean_answer} ({latency:.2f}с)")
                
                # Обратная связь
                feedback = ""
                while feedback not in ["да", "нет"]:
                    feedback = input("ℹ️ Вас устраивает ответ? (да/нет): ").lower()
                    if feedback not in ["да", "нет"]:
                        print("Введите 'да' или 'нет'")
                
                if feedback == "да":
                    success = True
                    clarification_used = attempt_num > 1
                    break
                else:
                    if attempt_num < max_attempts:
                        clarification = input("🔄 Уточните вопрос: ").strip()
                        current_question += f" ({clarification})"
        
            except Exception as e:
                stop_animation.set()
                animation_thread.join()
                print(f"\rОшибка генерации: {e}")
                answer = "Не удалось сгенерировать ответ"
                break
        
        # Создание записи
        record = {
            "original_question": original_question,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "attempts": attempts,
            "status": "clarifications" if clarification_used else "success" if success else "operator_required"
        }
        
        # Вывод статуса
        if success:
            if clarification_used:
                print(f"⚠️ Ответ с уточнениями сохранен в истории (попыток: {attempt_num})")
            else:
                print("✅ Ответ сохранен в истории успешных")
        else:
            print("❌ Бот: Передаю запрос оператору")
        
        qa_history.append(record)
        
        # Сохранение в файл
        with open("rugpt_qa_full_history.json", "w", encoding="utf-8") as f:
            json.dump(qa_history, f, ensure_ascii=False, indent=2)

    print("Бот: До свидания!")

In [40]:
# Запуск чат-бота
if __name__ == "__main__":
    chat_bot()

Бот: Здравствуйте! Для выхода введите 'выход'.



Вы:  кто президент РФ с 2012 года?


Бот: А. И. Шувалов, В. С. Черномырдин, М. Ф. Шмаков, Н. П. Кожедубов и МН. Зарубинский являются президентами России и Казахстана с 2011 года. (2.47с)


ℹ️ Вас устраивает ответ? (да/нет):  нет
🔄 Уточните вопрос:  напиши полное имя и фамилию президента Российской Федерации


Бот: В 2012 году президент России Владимир Путин подписал Указ о создании комиссии по совершенствованию деятельности федеральных органов исполнительной власти, осуществляющих функции по осуществлению государственного контроля и надзора в сфере экономики, промышленной политики и регулирования производства и обращения лекарственных средств и медицинских изделий, а также (2.61с)


...

Бот: Первой мировой войной в Австрии, как и во многих других странах Европы и Азии, завершилась Вторая Мировая война 1914—1918 годов, когда австрийские войска вторглись в Восточную Пруссию и Восточно-Франкскую Поморскую область, (2.67с)


ℹ️ Вас устраивает ответ? (да/нет):  да


⚠️ Ответ с уточнениями сохранен в истории (попыток: 2)



Вы:  выход


Бот: До свидания!
